# Document Loaders & Text Splitters

Before we can do RAG, we need to:

1. Load documents from various sources  
2. Split them into smaller chunks  

### Step 1: Load documents
- PDF
- TXT
- WEB
- Other sources

### Step 2: Split documents
Break documents into small chunks.

### Step 3: Store chunks
Later used in search.

## Why Split?

LLMs have limited context windows.  
We split documents into smaller chunks, store them, and retrieve only the relevant chunks at query time.

```text
Load → Split → Store → Retriever
```

## Document Loader

We convert file into Document Object, Langchain work with Document format, Loader converts  raw file `->` into structured document

| Loader Name | What It Loads | Import Statement |
|---|---|---|
| PyPDFLoader | PDF files | `from langchain_community.document_loaders import PyPDFLoader` |
| TextLoader | TXT / plain text files | `from langchain_community.document_loaders import TextLoader` |
| CSVLoader | CSV files | `from langchain_community.document_loaders import CSVLoader` |
| UnstructuredWordDocumentLoader | DOCX / Word files | `from langchain_community.document_loaders import UnstructuredWordDocumentLoader` |
| UnstructuredExcelLoader | Excel files (.xlsx) | `from langchain_community.document_loaders import UnstructuredExcelLoader` |
| JSONLoader | JSON files | `from langchain_community.document_loaders import JSONLoader` |
| DirectoryLoader | Load all files from a directory | `from langchain_community.document_loaders import DirectoryLoader` |
| WebBaseLoader | Web pages / URLs | `from langchain_community.document_loaders import WebBaseLoader` |
| UnstructuredHTMLLoader | HTML files | `from langchain_community.document_loaders import UnstructuredHTMLLoader` |
| MarkdownLoader | Markdown (.md) files | `from langchain_community.document_loaders import UnstructuredMarkdownLoader` |
| PythonLoader | Python source code (.py) | `from langchain_community.document_loaders import PythonLoader` |
| NotebookLoader | Jupyter notebooks (.ipynb) | `from langchain_community.document_loaders import NotebookLoader` |
| BSHTMLLoader | HTML using BeautifulSoup | `from langchain_community.document_loaders import BSHTMLLoader` |
| WikipediaLoader | Wikipedia articles | `from langchain_community.document_loaders import WikipediaLoader` |
| YoutubeLoader | YouTube transcripts | `from langchain_community.document_loaders import YoutubeLoader` |

### 1. TextLoader

In [1]:
from langchain_community.document_loaders import TextLoader

text_loader= TextLoader("../Data/Examples/example.txt", encoding="utf-8")

docs= text_loader.load()


print(f" Loaded {len(docs)} documents")
print(f" Document Type {type(docs[0])}")
print(f" Content Preview:  {docs[0].page_content[:100]}...")
print(f"Metadata: {docs[0].metadata}")

 Loaded 1 documents
 Document Type <class 'langchain_core.documents.base.Document'>
 Content Preview:  
What are Document Loaders?
Document Loader is one of the components of the LangChain framework. It ...
Metadata: {'source': '../Data/Examples/example.txt'}


### 2. PyPDFLoader

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_loader= PyPDFLoader("../Data/Examples/example.pdf")
docs= pdf_loader.load()


print(f" Loaded {len(docs)} documents")
print(f" Document Type {type(docs[0])}")
print(f" Content Preview:  {docs[0].page_content[:100]}...")
print(f"Metadata: {docs[0].metadata}")

 Loaded 5 documents
 Document Type <class 'langchain_core.documents.base.Document'>
 Content Preview:  What are Document Loaders?
Document Loader is one of the components of the LangChain framework. It i...
Metadata: {'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'source': '../Data/Examples/example.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


### 3. WebBaseLoader

In [3]:
import os
import bs4
from langchain_community.document_loaders import WebBaseLoader

os.environ["USER_AGENT"] = "my-langchain-app"

url = "https://en.wikipedia.org/wiki/LangChain"

web_loader = WebBaseLoader(
    web_paths=[url],
    bs_kwargs={
        "parse_only":bs4.SoupStrainer(id="mw-content-text")
    }
    )


docs = web_loader.load()


print(f" Loaded {len(docs)} documents")
print(f" Document Type {type(docs[0])}")
print(f" Content Preview:  {docs[0].page_content[:100]}...")
print(f"Metadata: {docs[0].metadata}")


USER_AGENT environment variable not set, consider setting it to identify your requests.


 Loaded 1 documents
 Document Type <class 'langchain_core.documents.base.Document'>
 Content Preview:  
Language model application development framework
LangChainDeveloperHarrison ChaseInitial releaseOct...
Metadata: {'source': 'https://en.wikipedia.org/wiki/LangChain'}


# Text Splitters

## Basic Idea
We break big document into small parts
These parts are called **chunks**.

## Core Concept

Big Document --> Small Chunks --> better understanding

## Chunk Overlap (Very Important)

We repeat some part between chunks
So AI does not lose context

### Visual Understading

```python
raw_text : "AAAA BBBB CCCC DDDD EEEE FFFF GGGG"
chunk 1: [ AAAA BBBB CCCC]
chunk 2:             [CCCCC DDDD EEEE]
chunk 3:                         [EEEE FFFF GGGG]
```

# CharacterTextSplitter vs RecursiveCharacterTextSplitter

## CharacterTextSplitter

Uses only **one separator** to split text.

Example:

```python
CharacterTextSplitter(
    chunk_size=200,
    separator="\n"
)
```

### How it works
- Splits text only by newline (`\n`)
- Then combines pieces until chunk size reaches the limit

### Important Limitation
`chunk_size` is **not a strict guarantee**.

If a single split part is already larger than the given size, it cannot split further.

Example:
- `chunk_size = 200`
- One line length = 1500

Result:
- Chunk will still be 1500 characters
- Warning appears:

```text
Created a chunk of size 1520, which is longer than the specified 200
```

### Why this happens
Because the splitter is allowed to split only using the specified separator.

If no separator exists inside that large text block, it keeps the whole block unchanged.

---

# RecursiveCharacterTextSplitter

Uses multiple separators recursively.

Example:

```python
RecursiveCharacterTextSplitter(
    chunk_size=200,
    separators=["\n\n", "\n", ".", " ", ""]
)
```

## How it works

It tries smarter splitting step by step:

1. First by paragraph (`\n\n`)
2. Then by line (`\n`)
3. Then by sentence (`.`)
4. Then by word (` `)
5. Finally by character (`""`)

If one method cannot keep the chunk under the limit, it automatically tries the next smaller separator.

---

# Why Recursive Splitter Is Better

✅ Better chunk size control  
✅ Fewer oversized chunks  
✅ Preserves context more naturally  
✅ Works well with messy text  
✅ Preferred for RAG and embeddings

---

# Main Difference

| Feature | Character | Recursive |
|---|---|---|
| Separator Count | One | Multiple |
| Smart Fallback | ❌ No | ✅ Yes |
| Exact Chunk Size | Not guaranteed | Much better control |
| Oversized Chunk Problem | Common | Rare |
| Best For | Simple text | RAG / LLM apps |

---

# Key Point

Even if you set:

```python
chunk_size = 200
```

it does **not mean every chunk will be exactly 200 characters**.

It means:

> “Try to keep chunks around this size if possible.”

Only `RecursiveCharacterTextSplitter` can enforce this more effectively because it has multiple fallback splitting strategies.

In [4]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap= 30,
    separators= [""] # "\n\n", "\n", ".", " ", 
)

chunks= splitter.split_documents(docs)
print(f"Original docs: {len(docs)}")
print(f"After Splitting: {len(chunks)} chunks")


Original docs: 1
After Splitting: 94 chunks


In [5]:
for i , chunk in enumerate(chunks):
    print(f"Chunks--> {i+1} ---> \n {chunk.page_content}")
    if i==4:
        break

Chunks--> 1 ---> 
 Language model application development framework
LangChainDeveloperHarrison ChaseInitial releaseOctober 2022Stable release0.1.16[1]
   / 11 April 2024; 2 years ago (11 April 2024)
Written inPython an
Chunks--> 2 ---> 
 pril 2024)
Written inPython and JavaScriptTypeSoftware framework for large language model application developmentLicenseMIT LicenseWebsiteLangChain.comRepositorygithub.com/langchain-ai/langchain

Free
Chunks--> 3 ---> 
 m/langchain-ai/langchain

Free and open-source software portal
LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language
Chunks--> 4 ---> 
 o applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and
Chunks--> 5 ---> 
 summarization, chatbots, and code analysis.[2]


History[edit]
LangChain was launched in October 2022 as

In [12]:

chara_splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap= 0,
    separator= "\n" # "\n\n", "\n", ".", " ", 
)

chunks= chara_splitter.split_documents(docs)
print(f"Original docs: {len(docs)}")
print(f"After Splitting: {len(chunks)} chunks")

Created a chunk of size 324, which is longer than the specified 200
Created a chunk of size 394, which is longer than the specified 200
Created a chunk of size 355, which is longer than the specified 200
Created a chunk of size 612, which is longer than the specified 200
Created a chunk of size 1520, which is longer than the specified 200
Created a chunk of size 217, which is longer than the specified 200
Created a chunk of size 202, which is longer than the specified 200
Created a chunk of size 202, which is longer than the specified 200


Original docs: 1
After Splitting: 79 chunks


In [13]:
chunks[:5]

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/LangChain'}, page_content='Language model application development framework\nLangChainDeveloperHarrison ChaseInitial releaseOctober 2022Stable release0.1.16[1]\n   / 11 April 2024; 2 years ago\xa0(11 April 2024)'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/LangChain'}, page_content='Written inPython and JavaScriptTypeSoftware framework for large language model application developmentLicenseMIT LicenseWebsiteLangChain.comRepositorygithub.com/langchain-ai/langchain'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/LangChain'}, page_content='Free and open-source software portal'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/LangChain'}, page_content="LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models